In [1]:
import requests
import pandas as pd
import numpy as np
import time

In [2]:
HEADERS = {
    "User-Agent": "your_name your_email@example.com"
}

In [3]:
def get_companyfacts(cik):
    cik = str(cik).zfill(10)
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()

    return r.json()

In [4]:
CCL= get_companyfacts(815097)

In [5]:
def flatten_companyfacts(companyfacts, symbol=None, taxonomy="us-gaap"):
    rows = []

    cik = companyfacts.get("cik")
    entity_name = companyfacts.get("entityName")

    facts = companyfacts.get("facts", {}).get(taxonomy, {})

    for tag, tag_data in facts.items():
        label = tag_data.get("label")
        description = tag_data.get("description")
        units = tag_data.get("units", {})

        for unit, observations in units.items():
            for obs in observations:
                row = {
                    "symbol": symbol,
                    "cik": cik,
                    "entity_name": entity_name,
                    "tag": tag,
                    "label": label,
                    "description": description,
                    "unit": unit,
                    **obs
                }
                rows.append(row)

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    # Dates
    for col in ["start", "end", "filed"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # Keep 10-K and 10-Q only
    df = df[df["form"].isin(["10-K", "10-Q"])].copy()

    # Useful period length
    df["period_days"] = (df["end"] - df["start"]).dt.days

    # Clean numeric value
    df["val"] = pd.to_numeric(df["val"], errors="coerce")

    return df

In [6]:
CCL_flattened= flatten_companyfacts(CCL)

In [7]:
CCL_flattened.describe

<bound method NDFrame.describe of       symbol     cik           entity_name  \
0       None  815097  Carnival Corporation   
1       None  815097  Carnival Corporation   
2       None  815097  Carnival Corporation   
3       None  815097  Carnival Corporation   
4       None  815097  Carnival Corporation   
...      ...     ...                   ...   
17785   None  815097  Carnival Corporation   
17786   None  815097  Carnival Corporation   
17787   None  815097  Carnival Corporation   
17788   None  815097  Carnival Corporation   
17789   None  815097  Carnival Corporation   

                                                     tag  \
0                                 AccountsPayableCurrent   
1                                 AccountsPayableCurrent   
2                                 AccountsPayableCurrent   
3                                 AccountsPayableCurrent   
4                                 AccountsPayableCurrent   
...                                                  

In [8]:
def clean_facts(df):
    df = df.copy()

    # Only USD values
    df = df[df["unit"] == "USD"]

    # Only 10-K and 10-Q
    df = df[df["form"].isin(["10-K", "10-Q"])]

    # Drop duplicates (keep latest filing)
    df = (
        df.sort_values(["tag", "start", "end", "filed"])
          .drop_duplicates(subset=["tag", "start", "end"], keep="last")
    )

    return df

In [9]:
CCL_clean_1= clean_facts(CCL_flattened).head(2)

In [10]:
def filter_ttm_window(df, years_back=2, buffer_days=360):
    df = df.copy()

    df["end"] = pd.to_datetime(df["end"], errors="coerce")

    cutoff_date = (
        pd.Timestamp.today().normalize()
        - pd.DateOffset(years=years_back)
        - pd.Timedelta(days=buffer_days)
    )

    df = df[df["end"].notna()].copy()
    df = df[df["end"] >= cutoff_date].copy()

    return df

In [11]:
CCL_clean_1 = clean_facts(CCL_flattened)

CCL_clean_2 = filter_ttm_window(CCL_clean_1)

CCL_clean_2.head()

,symbol,cik,entity_name,tag,label,description,unit,end,val,accn,fy,fp,form,filed,frame,start,period_days
116,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2023-05-31,1.042000e+09,0000815097-23-000051,2023.0,Q2,10-Q,2023-06-28,CY2023Q2I,NaT,NaN
117,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2023-08-31,1.103000e+09,0000815097-23-000066,2023.0,Q3,10-Q,2023-09-29,CY2023Q3I,NaT,NaN
122,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2023-11-30,1.168000e+09,0000815097-25-000007,2024.0,FY,10-K,2025-01-27,CY2023Q4I,NaT,NaN
123,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2024-02-29,1.103000e+09,0000815097-24-000035,2024.0,Q1,10-Q,2024-03-27,CY2024Q1I,NaT,NaN
124,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2024-05-31,1.063000e+09,0000815097-24-000057,2024.0,Q2,10-Q,2024-06-27,CY2024Q2I,NaT,NaN


In [12]:
CCL_clean_1 = clean_facts(CCL_flattened)
CCL_clean_2 = filter_ttm_window(CCL_clean_1)

print(CCL_clean_1.shape)
print(CCL_clean_2.shape)

(8310, 17)
(1475, 17)


In [13]:
FCF_INCLUDE_KEYWORDS = [
    "profit",
    "net income",
    "operating income",
    "income tax",
    "interest expense",
    "depreciation",
    "amortization",
    "capital expenditure",
    "property plant equipment",
    "accounts receivable",
    "receivables",
    "inventory",
    "accounts payable",
    "payables",
    "deferred revenue",
    "customer deposits",
    "debt",
    "borrowings",
    "repayments",
    "operating activities",
    "investing activities",
    "financing activities",
]

In [14]:
def split_known_fcf_items(df, keywords=FCF_INCLUDE_KEYWORDS):
    df = df.copy()

    search_text = (
        df["tag"].fillna("").astype(str) + " " +
        df["label"].fillna("").astype(str) + " " +
        df["description"].fillna("").astype(str)
    ).str.lower()

    mask = pd.Series(False, index=df.index)

    for keyword in keywords:
        mask = mask | search_text.str.contains(keyword.lower(), regex=False)

    known_fcf_items = df[mask].copy()
    manual_review_items = df[~mask].copy()

    return known_fcf_items, manual_review_items

In [15]:
CCL_known_fcf_items, CCL_manual_review_items = split_known_fcf_items(CCL_clean_2)

In [16]:
print(CCL_manual_review_items.shape)
print(CCL_known_fcf_items.shape)

(808, 17)
(667, 17)


In [17]:
CCL_manual_unique_tags = (
    CCL_manual_review_items[
        ["tag", "label", "description", "unit"]
    ]
    .drop_duplicates(subset=["tag"])
    .sort_values("tag")
    .reset_index(drop=True)
)

CCL_manual_unique_tags

,tag,label,description,unit
0,AccruedLiabilitiesCurrent,"Accrued Liabilities, Current",Carrying value as of the balance sheet date of...,USD
1,AccumulatedOtherComprehensiveIncomeLossDefined...,"Accumulated Other Comprehensive (Income) Loss,...","Amount, after tax, of accumulated other compre...",USD
2,AccumulatedOtherComprehensiveIncomeLossForeign...,"Accumulated Other Comprehensive Income (Loss),...","Accumulated adjustment, net of tax, that resul...",USD
3,AdvertisingExpense,Advertising Expense,Amount charged to advertising expense for the ...,USD
4,AociLossCashFlowHedgeCumulativeGainLossAfterTax,"AOCI, Cash Flow Hedge, Cumulative Gain (Loss),...","Amount, after tax, of accumulated gain (loss) ...",USD
...,...,...,...,...
97,UnrecordedUnconditionalPurchaseObligationBalan...,"Unrecorded Unconditional Purchase Obligation, ...",Amount of fixed and determinable portion of un...,USD
98,UnrecordedUnconditionalPurchaseObligationBalan...,"Unrecorded Unconditional Purchase Obligation, ...",Amount of fixed and determinable portion of un...,USD
99,UnrecordedUnconditionalPurchaseObligationBalan...,Unrecorded Unconditional Purchase Obligation,Amount of the unrecorded obligation to transfe...,USD
100,UnrecordedUnconditionalPurchaseObligationDueAf...,"Unrecorded Unconditional Purchase Obligation, ...",Amount of fixed and determinable portion of un...,USD


In [18]:
tag_counts = (
    CCL_manual_review_items["tag"]
    .value_counts()
    .rename("count")
    .reset_index()
    .rename(columns={"index": "tag"})
)

CCL_manual_unique_tags = CCL_manual_unique_tags.merge(
    tag_counts,
    on="tag",
    how="left"
)

CCL_manual_unique_tags.sort_values("count", ascending=False)

,tag,label,description,unit,count
68,OtherComprehensiveIncomeOtherNetOfTax,"Other Comprehensive Income, Other, Net of Tax",Amount of increase (decrease) in other compreh...,USD,18
47,InterestExpenseNonoperating,None,None,USD,18
58,OperatingCostsAndExpenses,Operating Costs and Expenses,Generally recurring costs associated with norm...,USD,18
67,OtherComprehensiveIncomeLossNetOfTaxPortionAtt...,"Other Comprehensive Income (Loss), Net of Tax,...",Amount after tax of other comprehensive income...,USD,18
66,OtherComprehensiveIncomeLossNetOfTax,"Other Comprehensive Income (Loss), Net of Tax",Amount after tax and reclassification adjustme...,USD,18
...,...,...,...,...,...
27,DerivativeFairValueOfDerivativeLiability,"Derivative Liability, Fair Value, Gross Liability","Fair value, before effects of master netting a...",USD,1
22,DerivativeAssetFairValueGrossLiability,"Derivative Asset, Fair Value, Gross Liability",Fair value of liability associated with financ...,USD,1
23,DerivativeAssets,Derivative Asset,"Fair value, after the effects of master nettin...",USD,1
72,PaymentsToAcquireInterestInSubsidiariesAndAffi...,Payments to Acquire Interest in Subsidiaries a...,The cash outflow associated with the acquisiti...,USD,1


In [20]:
tag_magnitude = (
    CCL_manual_review_items
    .assign(abs_val=lambda x: x["val"].abs())
    .groupby("tag")["abs_val"]
    .max()
    .rename("max_abs_value")
    .reset_index()
)

CCL_manual_unique_tags = CCL_manual_unique_tags.merge(
    tag_magnitude,
    on="tag",
    how="left"
)

In [21]:
CCL_manual_unique_tags = CCL_manual_unique_tags[
    ["tag", "label", "description", "unit", "count", "max_abs_value"]
].sort_values("max_abs_value", ascending=False)

CCL_manual_unique_tags

,tag,label,description,unit,count,max_abs_value
56,LiabilitiesAndStockholdersEquity,Liabilities and Equity,"Amount of liabilities and equity items, includ...",USD,12,5.187300e+10
6,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,12,5.187300e+10
82,RevenueFromContractWithCustomerExcludingAssess...,"Revenue from Contract with Customer, Excluding...","Amount, excluding tax collected from customer,...",USD,18,2.662200e+10
18,CostsAndExpenses,Costs and Expenses,Total costs of sales and operating expenses fo...,USD,14,2.144700e+10
58,OperatingCostsAndExpenses,Operating Costs and Expenses,Generally recurring costs associated with norm...,USD,18,1.594700e+10
...,...,...,...,...,...,...
22,DerivativeAssetFairValueGrossLiability,"Derivative Asset, Fair Value, Gross Liability",Fair value of liability associated with financ...,USD,1,0.000000e+00
39,ImpairmentOfLongLivedAssetsHeldForUse,Impairment of Long-Lived Assets Held-for-use,The aggregate amount of write-downs for impair...,USD,1,0.000000e+00
74,PaymentsToAcquireShortTermInvestments,Payments to Acquire Short-term Investments,The cash outflow for securities or other asset...,USD,4,0.000000e+00
31,DerivativeLiabilityFairValueGrossAsset,"Derivative Liability, Fair Value, Gross Asset",Fair value of asset associated with financial ...,USD,1,0.000000e+00
